# Receipt dataset exploration

Sanity checks before training: label balance, token-length distribution, and where each field tends to sit on the page. Loads the committed annotation samples; point `ANN_DIR` at the full set to reproduce the numbers used to size `max_length`.

In [ ]:
import json, glob, os, sys
from collections import Counter
sys.path.insert(0, os.path.abspath('../src'))
from receipt_ner.labels import ID2LABEL, entity_of

ANN_DIR = '../data/annotations'
records = [json.load(open(p)) for p in sorted(glob.glob(f'{ANN_DIR}/*.json'))]
print(f'{len(records)} annotated receipts loaded')

## Label balance

As expected for KIE, `O` dominates — this is why we score entity-level F1 with seqeval rather than token accuracy.

In [ ]:
counts = Counter()
for r in records:
    for tag in r['ner_tags']:
        counts[ID2LABEL[tag]] += 1
for label, n in counts.most_common():
    print(f'{label:>14}  {n}')

entity_counts = Counter(entity_of(ID2LABEL[t]) for r in records for t in r['ner_tags'] if entity_of(ID2LABEL[t]))
print('\nentities:', dict(entity_counts))

## Token count per receipt

Drives the choice of `max_length=512` (comfortably above the longest receipt after BPE splitting).

In [ ]:
lengths = [len(r['words']) for r in records]
print('words/receipt  min/mean/max:', min(lengths), sum(lengths)/len(lengths), max(lengths))

## Vertical position of the survey code

The code lives in the lower-middle band of the receipt (normalized y). This spatial regularity is exactly the signal LayoutLMv3's 2D position embeddings exploit.

In [ ]:
ys = []
for r in records:
    for tag, box in zip(r['ner_tags'], r['boxes']):
        if entity_of(ID2LABEL[tag]) == 'SURVEY_CODE':
            ys.append(box[1])  # normalized y0 (0-1000)
print('SURVEY_CODE y0 (0-1000):', ys)